In [ ]:
# ==========================================================
# セル1：ライブラリの読み込みと NR-500形式リーダの定義
# ==========================================================
# 【対象ファイル】1行目が "#BeginHeader,<行数>" で始まるCSV（KEYENCE NR-500）
#   Futaba形式（1行目が "Time:" で始まる）は 01/02/03 を使ってください。
# ==========================================================
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict

# ---- グラフの日本語文字化け対策（Windows最適化）----
plt.rcParams["font.family"] = ["Meiryo", "Yu Gothic", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*findfont: Font family.*not found.*"
)

# ==========================================================
# ★ 単位換算：圧力[MPa] = 電圧[V] × 20
#   NR-500 は「スケーリング OFF」で生電圧のまま記録されているため換算が必要。
#   係数を変える場合はこの1箇所だけ書き換えれば全体に反映されます。
# ==========================================================
VOLT_TO_MPA = 20.0


def probe_nr500(path):
    """先頭のヘッダだけを読んで NR-500形式か判定し、メタ情報を返す。

    NR-500形式でなければ None を返す（Futaba形式などはここで弾かれる）。
    """
    with open(path, "r", encoding="cp932", errors="replace") as f:
        head = [f.readline().rstrip("\n") for _ in range(120)]

    if not head[0].startswith("#BeginHeader"):
        return None

    # "#BeginHeader,71" → ヘッダ71行。列名行はその71行目なので skiprows=70
    n_header = int(head[0].split(",")[1])

    # "サンプリング周期,5μs" を秒に変換
    # （16行目の "実サンプリング周期" は先頭が「実」なので startswith で誤マッチしない）
    raw = next(l.split(",")[1] for l in head if l.startswith("サンプリング周期"))
    m = re.match(r"\s*([\d.]+)\s*(μs|us|ms|s)\s*$", raw)
    if m is None:
        raise ValueError(f"サンプリング周期を解釈できません: {raw!r}")
    dt = float(m.group(1)) * {"μs": 1e-6, "us": 1e-6, "ms": 1e-3, "s": 1.0}[m.group(2)]

    # "データ数,1000000" → これを nrows に使い、末尾3行のフッタ
    # （#BeginMark,3 / CH名,... / #EndMark）を構造的に除外する
    n_data = int(next(l.split(",")[1] for l in head if l.startswith("データ数")))

    return {"skiprows": n_header - 1, "dt": dt, "n_data": n_data}


def read_nr500(path, channels=("V03", "V04")):
    """NR-500形式CSVを読み、指定チャンネルを MPa に換算した DataFrame と dt を返す。"""
    meta = probe_nr500(path)
    if meta is None:
        raise ValueError("NR-500形式ではありません")

    header = pd.read_csv(
        path, skiprows=meta["skiprows"], encoding="cp932", nrows=0
    ).columns.tolist()

    # "V03" → "(1)HA-V03" を末尾一致で解決（ユニット番号が変わっても追従できる）
    resolved = {}
    for ch in channels:
        hit = [c for c in header if c.endswith(ch)]
        if not hit:
            raise ValueError(f"チャンネル '{ch}' が見つかりません（実際の列: {header}）")
        resolved[ch] = hit[0]

    read_kw = dict(
        skiprows=meta["skiprows"],
        encoding="cp932",
        usecols=list(resolved.values()),
        nrows=meta["n_data"],
    )
    try:
        # dtype を明示しないと pandas 3.0.3 では usecols 使用時に
        # IndexError: list index out of range が出るため、必ず指定する
        df = pd.read_csv(path, dtype="float32", **read_kw)
    except ValueError:
        # 数値以外が混入していた場合の保険
        df = pd.read_csv(path, dtype=str, **read_kw)
        df = df.apply(pd.to_numeric, errors="coerce").dropna().astype("float32")

    df = df.rename(columns={v: k for k, v in resolved.items()})[list(resolved)]
    return df * VOLT_TO_MPA, meta["dt"]


def spectrum(y, dt, f_lo, f_hi):
    """振幅スペクトルを計算し、表示帯域 f_lo〜f_hi だけに切り詰めて返す。

    NR-500 は 100万点あり rfft 後も 500,001点になる。全点を matplotlib に
    渡すと重ね描きで停止するため、★描画前に必ず帯域を切る★。
    0〜150Hz なら Δf=0.2Hz で 751点まで落ちる。
    """
    N = len(y)
    amp = np.abs(np.fft.rfft(y)) / (N / 2)
    amp[0] /= 2  # 直流成分だけは N で割る
    freq = np.fft.rfftfreq(N, d=dt)
    m = (freq >= f_lo) & (freq <= f_hi)
    return freq[m], amp[m]


def folder_tag(dirpath, main_dir):
    """main_dir からの相対パスをファイル名用のタグにする。

    末端フォルダ名だけだと 0.5mm/190℃ と 1.0mm/190℃ が同名になり
    上書きされてしまうため、相対パス全体を使って衝突を防ぐ。
    """
    rel = os.path.relpath(dirpath, main_dir)
    return "root" if rel == "." else rel.replace(os.sep, "_")


print("✅ ライブラリの読み込みと NR-500形式リーダの定義が完了しました。")
print(f"   単位換算: 電圧[V] × {VOLT_TO_MPA} = 圧力[MPa]")

In [ ]:
# ==========================================================
# セル2：メインフォルダの選択
# ==========================================================
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
main_dir = filedialog.askdirectory(title="解析対象のメインフォルダを選択してください")
root.destroy()

if not main_dir:
    print("⚠️ フォルダ選択がキャンセルされました。次のセルには進まず、やり直してください。")
else:
    print(f"✅ 選択されたメインフォルダ:\n{main_dir}")

In [ ]:
# ==========================================================
# セル3：対象ファイルの列挙と NR-500形式の絞り込み
# ==========================================================
if not main_dir:
    raise ValueError("メインフォルダが選択されていません。セル2を再実行してください。")

all_csv = []
for dirpath, dirnames, filenames in os.walk(main_dir):
    for f in filenames:
        if f.lower().endswith(".csv"):
            all_csv.append(os.path.join(dirpath, f))

# NR-500形式だけを対象にする。除外したファイルも件数と例を必ず表示し、
# 「黙って処理されていない」状態が起きないようにする。
target_files, skipped_files = [], []
for p in all_csv:
    try:
        (target_files if probe_nr500(p) is not None else skipped_files).append(p)
    except Exception as e:
        skipped_files.append(p)

print(f"🔍 見つかったCSVファイル: 合計 {len(all_csv)} 件")
print(f"   ├ NR-500形式 : {len(target_files)} 件 ← これを処理します")
print(f"   └ それ以外   : {len(skipped_files)} 件 ← 対象外として除外します")

if target_files:
    print("\n【処理対象ファイルの例】")
    for f in target_files[:3]:
        print(" -", f)
    if len(target_files) > 3:
        print(f"   ... (他 {len(target_files) - 3} 件)")
else:
    print("\n⚠️ NR-500形式のCSVが1件も見つかりませんでした。")
    print("   Futaba形式（1行目が 'Time:'）なら 01/02/03 を使ってください。")

if skipped_files:
    print("\n【対象外として除外したファイルの例】")
    for f in skipped_files[:3]:
        print(" -", os.path.basename(f))
    if len(skipped_files) > 3:
        print(f"   ... (他 {len(skipped_files) - 3} 件)")

In [ ]:
# ==========================================================
# セル4：1ファイル1枚のFFTグラフを一括生成して保存
# ==========================================================
if not target_files:
    raise ValueError("処理対象のNR-500形式ファイルがありません。セル3を確認してください。")

# ---- 表示設定 ----
CHANNELS = ["V03", "V04"]   # 解析するチャンネル（CH03/CH04 に相当）
F_LO, F_HI = 0, 150         # 表示する周波数帯域 [Hz]
F_STEP = 10                 # 横軸の目盛り間隔 [Hz]
# ------------------

output_dir_path = os.path.join(os.getcwd(), "fft_results_nr500")
os.makedirs(output_dir_path, exist_ok=True)

print("🚀 フーリエ変換を開始します...")
print(f"📁 画像の保存先フォルダ:\n  {output_dir_path}")
print(f"⏱️ 目安: 1ファイルあたり約0.6秒 → {len(target_files)} 件で約 "
      f"{len(target_files) * 0.6 / 60:.1f} 分\n")

success_count = 0
errors = []
t0 = time.time()

for i, file_path in enumerate(target_files, 1):
    csv_file = os.path.basename(file_path)
    try:
        df, dt = read_nr500(file_path, channels=CHANNELS)

        plt.figure(figsize=(10, 5))
        for ch in CHANNELS:
            y = df[ch].to_numpy(dtype=np.float64)
            freq, amp = spectrum(y, dt, F_LO, F_HI)
            plt.plot(freq, amp, label=ch, alpha=0.8, linewidth=1.0)

        plt.xlabel("Frequency [Hz]")
        plt.ylabel("Amplitude [MPa]")
        plt.title(f"FFT Spectrum - {csv_file}")
        plt.xlim(F_LO, F_HI)
        plt.xticks(np.arange(F_LO, F_HI + F_STEP, F_STEP))
        plt.legend()
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()

        # 末端フォルダ名だけだと板厚違いが衝突するため、相対パス全体をタグにする
        tag = folder_tag(os.path.dirname(file_path), main_dir)
        name = f"[{tag}]_{os.path.splitext(csv_file)[0]}_fft.png"
        plt.savefig(os.path.join(output_dir_path, name), dpi=150)
        plt.close()

        success_count += 1

    except Exception as e:
        plt.close()
        errors.append((file_path, f"{type(e).__name__}: {e}"))

    if i % 50 == 0 or i == len(target_files):
        el = time.time() - t0
        rest = el / i * (len(target_files) - i)
        print(f"  {i}/{len(target_files)} 件 完了  "
              f"（経過 {el:.0f}秒 / 残り約 {rest:.0f}秒）")

print("\n" + "=" * 40)
print("🏁 すべての処理が完了しました！")
print(f"✅ 成功: {success_count} 件")
print(f"⚠️ 失敗: {len(errors)} 件")
print("=" * 40)

if errors:
    print("\n【失敗したファイルの詳細】")
    for path, msg in errors:
        print(f" ❌ {os.path.basename(path)}: {msg}")